1.wconcept > best > 크롤링 [어떤 값을 크롤링 해와야 할 것인가? 고민]
2.크롤링 데이터 MySQL 전송
3.MySQL 전송된 데이터를 가지고 다음과 같은 결과를 출력 (SQL 문법 활용)

-판매가 기준 상위 20개 상품 조회
-할인율이 30% 이상인 상품 조회 & 내림차순 정렬
-특정 브랜드 (임의) 상품만 조회, 판매가 기준 오름차순 정렬
-리뷰수가 100이상인 상품의 평균 판매가 조회
-브랜드별 상품수 & 평균할인률 구하고 상품수가 많은 브랜드 10개 조회

크롤링할 정보:
브랜드명, 판매가격, 상품명, 할인율, 리뷰수

In [15]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time
import re

URL = "https://display.wconcept.co.kr/"

service = Service(ChromeDriverManager().install())
options = Options()

# options.add_argument("--headless")
# options.add_argument("--disable-gpu")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("--start-maximized")
options.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36")
options.add_argument("--lang=ko_KR")
options.add_argument("--no-sandbox")

driver = webdriver.Chrome(service=service, options=options)
driver.get(URL)
time.sleep(2)

best_btn = driver.find_elements(By.CSS_SELECTOR, " button.sc-d57aa1fb-0.iXIzbJ.btn.area-click")
best_btn[1].click()
time.sleep(2)

product_data = list()
products = driver.find_elements(By.CSS_SELECTOR,"div span.sc-228671f3-6.cWZeRh.area-info")
for index, product in enumerate(products[:50]) :
    try:
        brand = product.find_element(By.CSS_SELECTOR, "span.sc-5ee6d09d-0.hfpiuD.text.title").text.strip()
        product_name = product.find_element(By.CSS_SELECTOR, "span.sc-5ee6d09d-0.hfpiuD.text.detail").text.strip()
        product_price = product.find_element(By.CSS_SELECTOR, "span.sc-5ee6d09d-0.hfpiuD.text.final-price").text.strip()
        dis_rate = product.find_element(By.CSS_SELECTOR, "span.sc-5ee6d09d-0.hfpiuD.text.final-discount").text.strip()
        reviews_count= product.find_element(By.CSS_SELECTOR, "span.sc-5ee6d09d-0.hfpiuD.text.cnt").text.strip()
      
        
        product_data.append(f" {index +1},  브랜드명: {brand}, 상품명: {product_name}, 판매가격: {product_price}, 할인율: {dis_rate}, 리뷰수: {reviews_count}")
    except: 
        dis_rate = ""
        reviews_count = ""
print(product_data)



    
driver.quit()

[' 1,  브랜드명: 온앤온, 상품명: 스웨이드 봄버 점퍼 NEL6XM801_95, 판매가격: 242,592, 할인율: 39%, 리뷰수: (57)', ' 2,  브랜드명: 시야쥬, 상품명: [한정특가] SIOT4079 페이크 레더 봄버 자켓_Black, 판매가격: 123,530, 할인율: 34%, 리뷰수: (1,152)', ' 4,  브랜드명: 온앤온, 상품명: Suede Jacket NEL5XJ809_97, 판매가격: 239,590, 할인율: 33%, 리뷰수: (315)', ' 5,  브랜드명: 던스트, 상품명: ESSENTIAL 2 BUTTON WOOL BLAZER MELANGE GREY_UDJA6A202G2, 판매가격: 166,487, 할인율: 30%, 리뷰수: (88)', ' 6,  브랜드명: 오스트카카, 상품명: [14+5%쿠폰] [꼬민지 PICK][단독]트위드 싱글 롱 자켓_2colors, 판매가격: 215,100, 할인율: 10%, 리뷰수: (74)', ' 8,  브랜드명: 무니드, 상품명: [한정특가] PEARL OPEN COLLAR SHIRT - SILVER, 판매가격: 108,841, 할인율: 18%, 리뷰수: (1)', ' 9,  브랜드명: 무니드, 상품명: [한정특가] [단독] Scarf Blouse_2color, 판매가격: 90,801, 할인율: 23%, 리뷰수: (299)', ' 10,  브랜드명: 아티드, 상품명: [한정특가] 시에나 하프 스웨이드 자켓_BEIGE, 판매가격: 156,730, 할인율: 44%, 리뷰수: (11)', ' 11,  브랜드명: 드로우핏 우먼, 상품명: [20%쿠폰] 마그넷 버튼 와플 티셔츠 [3COLORS], 판매가격: 47,196, 할인율: 31%, 리뷰수: (1)', ' 12,  브랜드명: 레이허, 상품명: [한정특가] [그레이W컨셉 단독][ reorder]LAPEL V-KNIT [6COLORS], 판매가격: 58,996, 할인율: 32%, 리뷰수: (640)', ' 13,  브랜드명: 로브로브, 상품

In [ ]:
import pymysql

MYSQL_HOST = "127.0.0.1"
MYSQL_PORT = 3306
MYSQL_USER = "root"
MYSQL_PW = "Hayeon123!"
MYSQL_DB = "wconcept_crawling"
MYSQL_CHARSET = "utf8mb4"

rows = list()

for el in product_data :
    "brand" = el.get("brand")
    "product_name" = el.get("product_name")
    "product_price" = el.get("product_price")
    "dis_rate" = el.get("dis_rate")
    "reviews_count" = el.get("reviews_count")

    rows.append(brand,product_name,product_price,dis_rate,reviews_count)

print("DB INSERT 대상 row 수:", len(rows))

conn = pymysql.connect(
    host=MYSQL_HOST,
    port=MYSQL_PORT,
    user=MYSQL_USER,
    password=MYSQL_PW,
    database=MYSQL_DB,
    charset=MYSQL_CHARSET
)
#-----여기서부터해--- mysql에서 테이블 생성하는 것 부터 ㄱㄱ
try : 
    with conn.cursor() as cur :
        sql="""
        INSERT INTO ig_tag_comments (brand, post_url, author_id, comment_text) 
        VALUES (%s, %s, %s,%s);
        """

        for row in rows : 
            cur.execute(sql, row)
    
    conn.commit()
    print("MySQL 저장완료!")
    
except Exception as e :
    conn.rollback()
    print("저장실패: ", e)

finally :
    conn.close()
